# Autopoiesis through composition

_Investigation `autopoiesis-through-composition` — coder reproduction notebook._

**Question.** Can a precarious, self-bounding identity — a cell that produces its own boundary —
be assembled by composing existing parts, and how much of itself does each increment
produce? Progress is measured, not asserted: by operational closure (the autopoiesis
meter) and by precariousness (does the identity dissipate when self-production stops?).

A whole-cell model built up as a self-producing network, increment by increment.
The biological schema framework DRIVES acceptance: each study's verdict is the
autopoiesis meter (operational closure: gap = requires \ provides \ boundary) plus
the precariousness behavior test. The membrane — the self-produced boundary that
the whole-cell-modeling lineage models as a given — is the part we build.

---

This notebook re-runs each study with the workspace's own process-bigraph protocol and renders its figures. The text states the **question and parameters** only — the figures produced by each run are the results. Set `RERUN = False` in the setup cell to render the committed `runs.db` without re-simulating.


In [ ]:
"""Self-contained reproduction of this investigation.

Generated by vivarium-dashboard (notebook_export). Each study below is re-run
live with the workspace's own process-bigraph protocol and its figures are
rendered from the resulting runs.db.
"""
import os
import sys
from pathlib import Path

# Resolve the repository root robustly so this notebook runs from a fresh clone
# at ANY path with no setup (no env var, no path editing). Priority:
#   1. $VIVARIUM_REPO, if it points at a real directory;
#   2. walk up from the notebook's working directory for the repo markers
#      (a directory holding both 'workspace/' and 'pyproject.toml') — Jupyter
#      starts in the notebook's dir, so a committed notebook finds its own root;
#   3. the absolute path it was generated for (back-compat for old layouts);
#   4. the current working directory (last resort).
def _find_repo_root(_start):
    for _cand in (_start, *_start.parents):
        if (_cand / "workspace").is_dir() and (_cand / "pyproject.toml").is_file():
            return _cand
    return None

REPO = None
_env = os.environ.get("VIVARIUM_REPO")
if _env and Path(_env).is_dir():
    REPO = Path(_env)
if REPO is None:
    REPO = _find_repo_root(Path.cwd().resolve())
if REPO is None and Path('/home/runner/work/viva-autopoiesis/viva-autopoiesis').is_dir():
    REPO = Path('/home/runner/work/viva-autopoiesis/viva-autopoiesis')
if REPO is None:
    REPO = Path.cwd()
sys.path.insert(0, str(REPO))
# Composite specs use repo-root-relative paths (datasets, caches), and the
# workspace's runner/renderer assume cwd == repo root — so run from there.
os.chdir(REPO)

# Re-simulate from scratch? Set False to render the committed runs.db (fast).
RERUN = True

# --- standard process-bigraph protocol: register the workspace's Core ---
from pbg_autopoiesis.core import build_core
core = build_core()

# --- imported from the repo this notebook was generated for ---

from IPython.display import HTML, display

import contextlib as _contextlib, io as _io
@_contextlib.contextmanager
def quiet():
    """Silence the simulator's verbose per-step stdout so the notebook
    output stays readable (the figures below are the results)."""
    with _contextlib.redirect_stdout(_io.StringIO()):
        yield

import html as _htmlmod
def show_viz(_h, height=560):
    """Display a visualization's HTML in an isolated iframe.

    The figures embed their own scripts (e.g. Plotly); JupyterLab does not
    execute <script> tags from display(HTML(...)), so an iframe srcdoc is
    used instead — the browser runs the scripts inside the frame."""
    display(HTML(
        '<iframe srcdoc="{}" style="width:100%;height:{}px;border:0">'
        '</iframe>'.format(_htmlmod.escape(_h, quote=True), height)
    ))

import json as _json
def describe_spec(spec):
    """Print a composite spec's structure (parameters, processes, wiring)
    then the full editable dict. The spec is plain data — assign to any
    field (e.g. spec['state'][proc]['config'][...]) before building."""
    print("composite:", spec.get("name"))
    if spec.get("description"):
        print("description:", str(spec["description"]).strip())
    _params = spec.get("parameters") or {}
    if _params:
        print("\nparameters (filled into ${name} placeholders):")
        for _p, _pdef in _params.items():
            print(f"  {_p}: default={_pdef.get('default')!r}  type={_pdef.get('type')}")
    print("\nprocesses (node -> address):")
    for _node, _body in (spec.get("state") or {}).items():
        if not (isinstance(_body, dict) and _body.get("_type") == "process"):
            continue
        print(f"  {_node}  ->  {_body.get('address')}   interval={_body.get('interval')!r}")
        for _port in ("inputs", "outputs"):
            if _body.get(_port):
                print(f"      {_port} ports: {_body[_port]}")
    print("\nfull editable spec dict:")
    print(_json.dumps(spec, indent=2, default=str))

import base64 as _b64, pathlib as _pl
def _render_one(address, config, runs_db, study_yaml):
    """Generic figure renderer (no workspace render_study_viz.py):
    resolve an ``image:<relpath>`` visualization to displayable HTML,
    relative to the study directory."""
    addr = str(address or '')
    for _scheme in ('image:', 'file:', 'gif:', 'png:', 'svg:', 'jpg:', 'jpeg:'):
        if addr.startswith(_scheme):
            addr = addr[len(_scheme):]; break
    _p = _pl.Path(addr)
    if not _p.is_absolute():
        _p = _pl.Path(study_yaml).resolve().parent / _p
    if not _p.is_file():
        return f'<p style="color:#b91c1c">figure not found: {address}</p>'
    _suffix = _p.suffix.lower()
    if _suffix == '.svg':
        return _p.read_text(encoding='utf-8', errors='replace')
    if _suffix in ('.png', '.jpg', '.jpeg', '.gif', '.webp'):
        _mime = 'jpeg' if _suffix in ('.jpg', '.jpeg') else _suffix[1:]
        _data = _b64.b64encode(_p.read_bytes()).decode('ascii')
        return f'<img src="data:image/{_mime};base64,{_data}" style="max-width:100%"/>'
    if _suffix in ('.html', '.htm'):
        return _p.read_text(encoding='utf-8', errors='replace')
    return f'<p style="color:#6b7280">unsupported figure type: {address}</p>'

## Study: Study 1 — the minimal membrane/metabolism loop (`study-1-membrane-metabolism-loop`)

**Objective.** Assemble the smallest precarious, self-bounding identity by composing a toy metabolism,
a membrane that grows from its own lipids (and decays without them), and a boundary whose
volume is DERIVED from the membrane — then verify, with the autopoiesis meter, that the
network is operationally closed (modulo nutrient) and that the identity is precarious:
fed it persists, starved it dissipates.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `membrane-metabolism-loop` | `pbg_autopoiesis.composites.membrane-metabolism-loop` | 0 | supply_rate=2.0 |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `pbg_autopoiesis.composites.membrane-metabolism-loop`** — `spec_pbg_autopoiesis_composites_membrane_metabolism_loop` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_pbg_autopoiesis_composites_membrane_metabolism_loop = load_spec(REPO / 'pbg_autopoiesis/composites/membrane-metabolism-loop.composite.yaml')
describe_spec(spec_pbg_autopoiesis_composites_membrane_metabolism_loop)

In [ ]:
# === Edit parameters for composite 'membrane-metabolism-loop' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# tunable parameters (filled into ${name} placeholders):
spec_pbg_autopoiesis_composites_membrane_metabolism_loop['parameters']['supply_rate']['default'] = 2.0

# process 'supply'  (local:Supply)
spec_pbg_autopoiesis_composites_membrane_metabolism_loop['state']['supply']['interval'] = 1.0
spec_pbg_autopoiesis_composites_membrane_metabolism_loop['state']['supply']['config']['rate'] = '${supply_rate}'

# process 'metabolism'  (local:Metabolism)
spec_pbg_autopoiesis_composites_membrane_metabolism_loop['state']['metabolism']['interval'] = 1.0

# process 'membrane'  (local:Membrane)
spec_pbg_autopoiesis_composites_membrane_metabolism_loop['state']['membrane']['interval'] = 1.0

# process 'boundary'  (local:Boundary)
spec_pbg_autopoiesis_composites_membrane_metabolism_loop['state']['boundary']['interval'] = 1.0

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: study-1-membrane-metabolism-loop ===
STUDY = 'study-1-membrane-metabolism-loop'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**precariousness**


In [ ]:
# precariousness
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**dynamics**


In [ ]:
# dynamics
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**emergent_boundary**


In [ ]:
# emergent_boundary
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**volume_coupling**


In [ ]:
# volume_coupling
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**closure_cycle**


In [ ]:
# closure_cycle
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**phase_portrait**


In [ ]:
# phase_portrait
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| operational-closure | kind=derived_scalar field=closure_gap_size | op <= value 0 provenance {'kind': 'theory', 'note': 'gap = ∅ is the definitional autopoietic criterion (operational closure of constraints; Maturana-Varela, Montévil-Mossio): a network is closed iff every requirement is self-provided. The threshold is the theory, not a tuned value.'} |
| precariousness | kind=derived_scalar field=precariousness_ratio | op < value 0.3 provenance {'kind': 'calibration', 'note': '0.3 starved/fed ratio is a calibration band set so the self-producing loop (≈0.035) passes with wide margin while an externally-maintained mimic (≈9x persistent) fails — calibrated against the two controls, not a literature value.'} |
| identity-persists-when-fed | kind=derived_scalar field=fed_volume_growth | op >= value 1.0 |


## Study: Study 2 — spatial containment: holding the individual together (`study-2-spatial-containment`)

**Objective.** Study 1 had no space — "inside" was a scalar volume. In a medium the self-produced
interior would diffuse away and the individual would dissolve. Put the cell on a 1-D
lattice where interior content DIFFUSES and the self-produced membrane GATES the
boundary flux, and show that the membrane — built by the metabolism it contains —
is what holds the individual together against dispersion. The autopoiesis of
containment, with precariousness extended to space.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `spatial-containment` | `pbg_autopoiesis.composites.spatial-containment` | 0 | supply_rate=2.0 |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `pbg_autopoiesis.composites.spatial-containment`** — `spec_pbg_autopoiesis_composites_spatial_containment` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_pbg_autopoiesis_composites_spatial_containment = load_spec(REPO / 'pbg_autopoiesis/composites/spatial-containment.composite.yaml')
describe_spec(spec_pbg_autopoiesis_composites_spatial_containment)

In [ ]:
# === Edit parameters for composite 'spatial-containment' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# tunable parameters (filled into ${name} placeholders):
spec_pbg_autopoiesis_composites_spatial_containment['parameters']['fed']['default'] = True
spec_pbg_autopoiesis_composites_spatial_containment['parameters']['membrane_on']['default'] = True

# process 'spatial'  (local:SpatialContainment)
spec_pbg_autopoiesis_composites_spatial_containment['state']['spatial']['interval'] = 1.0
spec_pbg_autopoiesis_composites_spatial_containment['state']['spatial']['config']['fed'] = '${fed}'
spec_pbg_autopoiesis_composites_spatial_containment['state']['spatial']['config']['membrane_on'] = '${membrane_on}'

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: study-2-spatial-containment ===
STUDY = 'study-2-spatial-containment'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**containment_profiles**


In [ ]:
# containment_profiles
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**kymograph**


In [ ]:
# kymograph
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**containment_over_time**


In [ ]:
# containment_over_time
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| containment | kind=derived_scalar field=containment_ratio | op >= value 3.0 provenance {'kind': 'calibration', 'note': '3x interior/exterior concentration is a clear-separation band: the self-produced membrane case (~60x) passes with margin, calibrated so containment is unambiguous rather than marginal. Not a literature value.'} |
| membrane-counters-diffusion | kind=derived_scalar field=membrane_effect | op >= value 2.0 provenance {'kind': 'theory', 'note': 'The membrane must more than double containment vs metabolism-alone for the self-produced boundary (not diffusion physics) to be the cause; 2x is the minimal "membrane dominates" criterion implied by the discriminative claim.'} |
| spatial-precariousness | kind=derived_scalar field=precariousness_collapse | op < value 0.3 |


## Study: Study 3 — adaptive chemotaxis: move toward food to survive (`study-3-adaptive-chemotaxis`)

**Objective.** Place the precarious individual in an environment with a nutrient gradient and give
it a sensorimotor loop: sense the local nutrient, run-and-tumble (tumble less when
nutrient is rising), move. The nutrient feeds the metabolism that maintains the
membrane that keeps it alive. Show that a chemotactic agent climbs the gradient,
finds food, and survives — while a blind agent random-walks, starves, and dissolves.
This is where life becomes mind: the gradient is MEANINGFUL from the perspective of
the precarious identity (up-gradient = viable, down-gradient = dissolution). Value is
grounded in the cell's own viability, not assigned from outside — sense-making, and
agency in service of self-maintenance.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `chemotactic-agent` | `pbg_autopoiesis.composites.adaptive-chemotaxis` | 0 | supply_rate=2.0 |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `pbg_autopoiesis.composites.adaptive-chemotaxis`** — `spec_pbg_autopoiesis_composites_adaptive_chemotaxis` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_pbg_autopoiesis_composites_adaptive_chemotaxis = load_spec(REPO / 'pbg_autopoiesis/composites/adaptive-chemotaxis.composite.yaml')
describe_spec(spec_pbg_autopoiesis_composites_adaptive_chemotaxis)

In [ ]:
# === Edit parameters for composite 'adaptive-chemotaxis' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# tunable parameters (filled into ${name} placeholders):
spec_pbg_autopoiesis_composites_adaptive_chemotaxis['parameters']['chemotactic']['default'] = True
spec_pbg_autopoiesis_composites_adaptive_chemotaxis['parameters']['seed']['default'] = 0

# process 'chemotaxis'  (local:Chemotaxis)
spec_pbg_autopoiesis_composites_adaptive_chemotaxis['state']['chemotaxis']['interval'] = 1.0
spec_pbg_autopoiesis_composites_adaptive_chemotaxis['state']['chemotaxis']['config']['chemotactic'] = '${chemotactic}'
spec_pbg_autopoiesis_composites_adaptive_chemotaxis['state']['chemotaxis']['config']['seed'] = '${seed}'

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: study-3-adaptive-chemotaxis ===
STUDY = 'study-3-adaptive-chemotaxis'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**trajectories**


In [ ]:
# trajectories
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**survival**


In [ ]:
# survival
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**landscape**


In [ ]:
# landscape
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| chemotaxis-survival | kind=derived_scalar field=chemotaxis_survival | op >= value 0.6 |
| agency-advantage | kind=derived_scalar field=survival_advantage | op >= value 1.5 provenance {'kind': 'calibration', 'note': '1.5x advantage band set so sensing must clearly beat the blind random-walk control (~0.35 survival); calibrated against the negative control rather than a literature effect size (observed 2.33x ± 0.26 over 12 seeds).'} |
| sense-making | kind=derived_scalar field=gradient_advantage | op >= value 1.15 provenance {'kind': 'theory', 'note': '>1 means the agent experiences more food than chance; 1.15 is a minimal "the gradient is made to matter" margin above parity — a theory-motivated floor for sense-making, not a calibrated or literature value.'} |


## Study: Study 4 — growth & division: one individual becomes a heterogeneous population (`study-4-growth-division`)

**Objective.** The precarious individual grows as its autopoietic loop runs and, past a size
threshold, DIVIDES into two daughters. The membrane is partitioned with noise so
daughters differ in size; a heritable trait is inherited with mutation so lineages
DIVERSIFY over generations — one individual becomes a population of non-identical
individuals. Division is itself precarious: a daughter that inherits too little
membrane to maintain itself dissolves — reproduction has a viability cost. This
heterogeneous population is the substrate on which study 5's selection can act.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `growing-population` | `pbg_autopoiesis.composites.growth-division` | 0 | supply_rate=2.0 |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `pbg_autopoiesis.composites.growth-division`** — `spec_pbg_autopoiesis_composites_growth_division` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_pbg_autopoiesis_composites_growth_division = load_spec(REPO / 'pbg_autopoiesis/composites/growth-division.composite.yaml')
describe_spec(spec_pbg_autopoiesis_composites_growth_division)

In [ ]:
# === Edit parameters for composite 'growth-division' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# tunable parameters (filled into ${name} placeholders):
spec_pbg_autopoiesis_composites_growth_division['parameters']['supply']['default'] = 0.55
spec_pbg_autopoiesis_composites_growth_division['parameters']['seed']['default'] = 0

# process 'growth'  (local:GrowthDivision)
spec_pbg_autopoiesis_composites_growth_division['state']['growth']['interval'] = 1.0
spec_pbg_autopoiesis_composites_growth_division['state']['growth']['config']['supply'] = '${supply}'
spec_pbg_autopoiesis_composites_growth_division['state']['growth']['config']['seed'] = '${seed}'

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: study-4-growth-division ===
STUDY = 'study-4-growth-division'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**population**


In [ ]:
# population
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**diversification**


In [ ]:
# diversification
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**heterogeneity**


In [ ]:
# heterogeneity
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| reproduction | kind=derived_scalar field=final_population | op >= value 10 |
| heterogeneity | kind=derived_scalar field=composition_heterogeneity | op >= value 0.05 provenance {'kind': 'exploratory', 'note': 'Existence-level band (this is an exploratory study): the threshold asks only whether trait diversity emerges at all (spread > ~0), not a calibrated or literature-anchored magnitude.'} |
| division-precariousness | kind=derived_scalar field=division_mortality | op >= value 0.01 provenance {'kind': 'exploratory', 'note': 'Existence-level band: any non-zero daughter mortality demonstrates reproduction carries a viability cost. Exploratory threshold (does the cost occur at all?), not a calibrated rate.'} |


## Study: Adversarial probes — can the framework be fooled? (`study-5-adversarial-probes`)

**Objective.** Turn the demonstration into a test: challenge the autopoiesis framework with
systems that should NOT qualify, and check that the metric REJECTS them. Two
probes — an externally-maintained mimic (persistence supplied from outside, not
self-produced) and a network missing a self-production step (no membrane
producer). The study passes only if the metric rejects both, showing the
criteria discriminate genuine self-production from superficial similarity.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `adversarial-probes` | `pbg_autopoiesis.composites.membrane-metabolism-loop` | 0 | supply_rate=0.0 |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `pbg_autopoiesis.composites.membrane-metabolism-loop`** — `spec_pbg_autopoiesis_composites_membrane_metabolism_loop` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_pbg_autopoiesis_composites_membrane_metabolism_loop = load_spec(REPO / 'pbg_autopoiesis/composites/membrane-metabolism-loop.composite.yaml')
describe_spec(spec_pbg_autopoiesis_composites_membrane_metabolism_loop)

In [ ]:
# === Edit parameters for composite 'membrane-metabolism-loop' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# tunable parameters (filled into ${name} placeholders):
spec_pbg_autopoiesis_composites_membrane_metabolism_loop['parameters']['supply_rate']['default'] = 2.0

# process 'supply'  (local:Supply)
spec_pbg_autopoiesis_composites_membrane_metabolism_loop['state']['supply']['interval'] = 1.0
spec_pbg_autopoiesis_composites_membrane_metabolism_loop['state']['supply']['config']['rate'] = '${supply_rate}'

# process 'metabolism'  (local:Metabolism)
spec_pbg_autopoiesis_composites_membrane_metabolism_loop['state']['metabolism']['interval'] = 1.0

# process 'membrane'  (local:Membrane)
spec_pbg_autopoiesis_composites_membrane_metabolism_loop['state']['membrane']['interval'] = 1.0

# process 'boundary'  (local:Boundary)
spec_pbg_autopoiesis_composites_membrane_metabolism_loop['state']['boundary']['interval'] = 1.0

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: study-5-adversarial-probes ===
STUDY = 'study-5-adversarial-probes'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| rejects-external-maintenance | kind=derived_scalar field=external_maintenance_persistence | op >= value 3.0 provenance {'kind': 'calibration', 'note': 'The mimic must be ≥3x more persistent than the self-producing loop when starved to count as "rejected" (not precarious); calibrated against the self-producing-loop positive control, mirroring study-1\'s precariousness band.'} |
| rejects-broken-closure | kind=derived_scalar field=broken_network_gap | op > value 0 provenance {'kind': 'theory', 'note': 'Any non-empty closure gap means a required type is not self-produced, so the network is definitionally not operationally closed — the threshold is the autopoietic closure definition, not a tuned value.'} |
